## Notes on CAPSTONE Project

In [229]:
import pandas as pd
import numpy as np
import json
import math
import time

In [42]:
base = '/Users/isabellawoods/Documents/northwestern 25-26/CAPSTONE'
traffic = pd.read_csv(f'{base}/DATA/Traffic.csv')
visit   = pd.read_csv(f'{base}/DATA/Visitation.csv', skiprows = 2)
species = pd.read_csv(f'{base}/NPSpecies.csv')


/var/folders/10/1rxl90mx7ns4kmscq1658bbm0000gn/T/ipykernel_67596/506586999.py:4: DtypeWarning: Columns (22,27) have mixed types. Specify dtype option on import or set low_memory=False.
  species = pd.read_csv(f'{base}/NPSpecies.csv')


In [70]:
traffic.head()

,ParkName,UnitCode,ParkType,Region,TrafficCounter,Year,Month,TrafficCount,ParkNameTotal,UnitCodeTotal,ParkTypeTotal,RegionType,TrafficCountTotalLabel,YearTotal,TrafficCountTotal
0,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,TRAFFIC COUNT AT KNOB CREEK,2012,1,0,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,TRAFFIC COUNT AT KNOB CREEK,2012,0
1,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,TRAFFIC COUNT AT KNOB CREEK,2012,2,0,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,TRAFFIC COUNT AT KNOB CREEK,2012,0
2,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,TRAFFIC COUNT AT KNOB CREEK,2012,3,0,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,TRAFFIC COUNT AT KNOB CREEK,2012,0
3,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,TRAFFIC COUNT AT KNOB CREEK,2012,4,0,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,TRAFFIC COUNT AT KNOB CREEK,2012,0
4,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,TRAFFIC COUNT AT KNOB CREEK,2012,5,0,Abraham Lincoln Birthplace NHP,ABLI,National Historical Park,Southeast,TRAFFIC COUNT AT KNOB CREEK,2012,0


In [36]:
# the downloaded csv contained two tables appended to each other
# row of column headers in the middle of the df
visit.index[visit['Year'] == 'Reporting']
# park visits is visitors by park
parkvisits = visit.iloc[:24372, :]

# total visitation numbers across all parks by year
totalvisits = visit.iloc[24372:, :]
# set columns as row 24372
totalvisits.columns = totalvisits.iloc[0]
totalvisits = totalvisits.iloc[1:]

In [149]:
# select all unique park names
pv_parks = parkvisits.ParkName.unique()
traffic_name = traffic.ParkName.unique()
traffic_code = traffic.UnitCode.unique()
species_parks = species["Park Code"].unique()

# select parks only present in all 3
traffic_species = set(traffic_code) & set(species_parks)
traffic_visits = set(traffic_name) & set(pv_parks)
#common = set(common_codes) & set(common_names)
print(f"Parks in all three datasets: {len(traffic_species)}")

code_to_name = dict(zip(traffic.UnitCode, traffic.ParkName))
traffic_species_names = {code_to_name[code] for code in traffic_species if code in code_to_name}

common = set(traffic_species_names) & set(traffic_visits)

Parks in all three datasets: 210


In [94]:
parks = [
    'Abraham Lincoln Birthplace National Historical Park, US, KY',
    'Acadia National Park, US, ME',
    'Allegheny Portage Railroad National Historic Site, US, PA',
    'Amistad National Recreation Area, US, TX',
    'Apostle Islands National Lakeshore, US, WI',
    'Appomattox Court House National Historical Park, US, VA',
    'Arches National Park, US, UT',
    'Arkansas Post National Memorial, US, AR',
    'Assateague Island National Seashore, US, MD',
    'Badlands National Park, US, SD',
    'Bandelier National Monument, US, NM',
    'Big Bend National Park, US, TX',
    'Big Cypress National Preserve, US, FL',
    'Big Hole National Battlefield, US, MT',
    'Big South Fork National River and Recreation Area, US, TN',
    'Big Thicket National Preserve, US, TX',
    'Bighorn Canyon National Recreation Area, US, MT',
    'Biscayne National Park, US, FL',
    'Black Canyon of the Gunnison National Park, US, CO',
    'Blue Ridge Parkway, US, VA',
    'Booker T. Washington National Monument, US, VA',
    'Bryce Canyon National Park, US, UT',
    'Buffalo National River, US, AR',
    'Cabrillo National Monument, US, CA',
    'Camp Nelson National Monument, US, KY',
    'Canaveral National Seashore, US, FL',
    'Cane River Creole National Historical Park, US, LA',
    'Canyon de Chelly National Monument, US, AZ',
    'Canyonlands National Park, US, UT',
    'Cape Cod National Seashore, US, MA',
    'Cape Hatteras National Seashore, US, NC',
    'Cape Lookout National Seashore, US, NC',
    'Capitol Reef National Park, US, UT',
    'Capulin Volcano National Monument, US, NM',
    'Carlsbad Caverns National Park, US, NM',
    'Catoctin Mountain Park, US, MD',
    'Cedar Breaks National Monument, US, UT',
    'Chaco Culture National Historical Park, US, NM',
    'Charles Pinckney National Historic Site, US, SC',
    'Chattahoochee River National Recreation Area, US, GA',
    'Chesapeake and Ohio Canal National Historical Park, US, MD',
    'Chickamauga and Chattanooga National Military Park, US, GA',
    'Chickasaw National Recreation Area, US, OK',
    'City of Rocks National Reserve, US, ID',
    'Colonial National Historical Park, US, VA',
    'Colorado National Monument, US, CO',
    'Congaree National Park, US, SC',
    'Coronado National Memorial, US, AZ',
    'Cowpens National Battlefield, US, SC',
    'Crater Lake National Park, US, OR',
    'Craters of the Moon National Monument and Preserve, US, ID',
    'Cumberland Gap National Historical Park, US, KY',
    'Curecanti National Recreation Area, US, CO',
    'Cuyahoga Valley National Park, US, OH',
    'De Soto National Memorial, US, FL',
    'Death Valley National Park, US, CA',
    'Delaware Water Gap National Recreation Area, US, PA',
    'Denali National Park and Preserve, US, AK',
    'Devils Postpile National Monument, US, CA',
    'Devils Tower National Monument, US, WY',
    'Dinosaur National Monument, US, CO',
    'Eisenhower National Historic Site, US, PA',
    'El Malpais National Monument, US, NM',
    'El Morro National Monument, US, NM',
    'Everglades National Park, US, FL',
    'Fire Island National Seashore, US, NY',
    'Florissant Fossil Beds National Monument, US, CO',
    'Fort Caroline National Memorial, US, FL',
    'Fort Donelson National Battlefield, US, TN',
    'Fort Frederica National Monument, US, GA',
    'Fort Laramie National Historic Site, US, WY',
    'Fort Larned National Historic Site, US, KS',
    'Fort Matanzas National Monument, US, FL',
    'Fort Necessity National Battlefield, US, PA',
    'Fort Point National Historic Site, US, CA',
    'Fort Pulaski National Monument, US, GA',
    'Fort Raleigh National Historic Site, US, NC',
    'Fort Vancouver National Historic Site, US, WA',
    'Fort Washington Park, US, MD',
    'Fossil Butte National Monument, US, WY',
    'Fredericksburg and Spotsylvania National Military Park, US, VA',
    'Gateway National Recreation Area, US, NY',
    'Gauley River National Recreation Area, US, WV',
    'George Washington Birthplace National Monument, US, VA',
    'George Washington Carver National Monument, US, MO',
    'George Washington Memorial Parkway, US, VA',
    'Gettysburg National Military Park, US, PA',
    'Glacier National Park, US, MT',
    'Glen Canyon National Recreation Area, US, UT',
    'Golden Gate National Recreation Area, US, CA',
    'Grand Canyon National Park, US, AZ',
    'Grand Portage National Monument, US, MN',
    'Grand Teton National Park, US, WY',
    'Great Basin National Park, US, NV',
    'Great Sand Dunes National Park and Preserve, US, CO',
    'Great Smoky Mountains National Park, US, TN',
    'Greenbelt Park, US, MD',
    'Guadalupe Mountains National Park, US, TX',
    'Guilford Courthouse National Military Park, US, NC',
    'Gulf Islands National Seashore, US, FL',
    'Haleakala National Park, US, HI',
    'Harpers Ferry National Historical Park, US, WV',
    'Hawaii Volcanoes National Park, US, HI',
    'Home of Franklin D. Roosevelt National Historic Site, US, NY',
    'Homestead National Historical Park, US, NE',
    'Hopewell Culture National Historical Park, US, OH',
    'Hopewell Furnace National Historic Site, US, PA',
    'Horseshoe Bend National Military Park, US, AL',
    'Hot Springs National Park, US, AR',
    'Hubbell Trading Post National Historic Site, US, AZ',
    'Indiana Dunes National Park, US, IN',
    'Jean Lafitte National Historical Park and Preserve, US, LA',
    'Jewel Cave National Monument, US, SD',
    'John Day Fossil Beds National Monument, US, OR',
    'Johnstown Flood National Memorial, US, PA',
    'Joshua Tree National Park, US, CA',
    'Katahdin Woods and Waters National Monument, US, ME',
    'Katmai National Park and Preserve, US, AK',
    'Kenai Fjords National Park, US, AK',
    'Kennesaw Mountain National Battlefield Park, US, GA',
    'Kings Mountain National Military Park, US, SC',
    'LBJ Memorial Grove on the Potomac, US, VA',
    'Lake Mead National Recreation Area, US, NV',
    'Lake Meredith National Recreation Area, US, TX',
    'Lake Roosevelt National Recreation Area, US, WA',
    'Lassen Volcanic National Park, US, CA',
    'Lava Beds National Monument, US, CA',
    'Lewis and Clark National Historical Park, US, OR',
    'Lincoln Boyhood National Memorial, US, IN',
    'Little Bighorn Battlefield National Monument, US, MT',
    'Little River Canyon National Preserve, US, AL',
    'Lyndon B. Johnson National Historical Park, US, TX',
    'Mammoth Cave National Park, US, KY',
    'Manassas National Battlefield Park, US, VA',
    'Manzanar National Historic Site, US, CA',
    'Martin Van Buren National Historic Site, US, NY',
    'Mesa Verde National Park, US, CO',
    'Minute Man National Historical Park, US, MA',
    'Missouri National Recreational River, US, NE',
    'Mojave National Preserve, US, CA',
    'Monocacy National Battlefield, US, MD',
    'Montezuma Castle National Monument, US, AZ',
    'Moores Creek National Battlefield, US, NC',
    'Morristown National Historical Park, US, NJ',
    'Mount Rainier National Park, US, WA',
    'Mount Rushmore National Memorial, US, SD',
    'Natchez Trace Parkway, US, MS',
    'Natural Bridges National Monument, US, UT',
    'Navajo National Monument, US, AZ',
    'New River Gorge National Park and Preserve, US, WV',
    'Nez Perce National Historical Park, US, ID',
    'Ninety Six National Historic Site, US, SC',
    'North Cascades National Park, US, WA',
    'Obed Wild and Scenic River, US, TN',
    'Ocmulgee Mounds National Historical Park, US, GA',
    'Olympic National Park, US, WA',
    'Oregon Caves National Monument and Preserve, US, OR',
    'Organ Pipe Cactus National Monument, US, AZ',
    'Ozark National Scenic Riverways, US, MO',
    'Padre Island National Seashore, US, TX',
    'Palo Alto Battlefield National Historical Park, US, TX',
    'Pea Ridge National Military Park, US, AR',
    'Petersburg National Battlefield, US, VA',
    'Petrified Forest National Park, US, AZ',
    'Petroglyph National Monument, US, NM',
    'Pictured Rocks National Lakeshore, US, MI',
    'Pinnacles National Park, US, CA',
    'Pipe Spring National Monument, US, AZ',
    'Piscataway Park, US, MD',
    'Point Reyes National Seashore, US, CA',
    'Prince William Forest Park, US, VA',
    "Pu'ukohola Heiau National Historic Site, US, HI",
    'Redwood National Park, US, CA',
    'Richmond National Battlefield Park, US, VA',
    'Rocky Mountain National Park, US, CO',
    'Russell Cave National Monument, US, AL',
    'Saguaro National Park, US, AZ',
    'Saint-Gaudens National Historical Park, US, NH',
    'San Antonio Missions National Historical Park, US, TX',
    'San Juan Island National Historical Park, US, WA',
    'Santa Monica Mountains National Recreation Area, US, CA',
    'Saratoga National Historical Park, US, NY',
    'Scotts Bluff National Monument, US, NE',
    'Shenandoah National Park, US, VA',
    'Shiloh National Military Park, US, TN',
    'Sleeping Bear Dunes National Lakeshore, US, MI',
    'Stones River National Battlefield, US, TN',
    'Sunset Crater Volcano National Monument, US, AZ',
    'Tallgrass Prairie National Preserve, US, KS',
    'Theodore Roosevelt National Park, US, ND',
    'Timucuan Ecological and Historic Preserve, US, FL',
    'Tonto National Monument, US, AZ',
    'Tuzigoot National Monument, US, AZ',
    'Upper Delaware Scenic and Recreational River, US, NY',
    'Valles Caldera National Preserve, US, NM',
    'Valley Forge National Historical Park, US, PA',
    'Vanderbilt Mansion National Historic Site, US, NY',
    'Vicksburg National Military Park, US, MS',
    'Walnut Canyon National Monument, US, AZ',
    'War in the Pacific National Historical Park, US, GU',
    'Washita Battlefield National Historic Site, US, OK',
    'Whiskeytown National Recreation Area, US, CA',
    'White Sands National Park, US, NM',
    'Whitman Mission National Historic Site, US, WA',
    "Wilson's Creek National Battlefield, US, MO",
    'Wind Cave National Park, US, SD',
    'Wright Brothers National Memorial, US, NC',
    'Yellowstone National Park, US, WY',
    'Yosemite National Park, US, CA',
    'Zion National Park, US, UT',
]

In [113]:
import requests 
from bs4 import BeautifulSoup

iNaturalist = "https://api.inaturalist.org/v1/places"
response = requests.get(iNaturalist)
soup = BeautifulSoup(response.text)
soup.get_text()

'{"total_results":338729102,"page":1,"per_page":30,"results":[{"quality_grade":"casual","time_observed_at":null,"taxon_geoprivacy":null,"annotations":[],"uuid":"45f11dc0-4778-4895-82b8-bca08158fff4","observed_on_details":{"date":"2026-04-17","day":17,"month":4,"year":2026,"hour":0,"week":16},"id":351512614,"cached_votes_total":0,"identifications_most_agree":false,"created_at_details":{"date":"2026-04-20","day":20,"month":4,"year":2026,"hour":13,"week":17},"species_guess":"Northern Cottonmouth","identifications_most_disagree":false,"tags":[],"positional_accuracy":1018,"comments_count":0,"site_id":1,"created_time_zone":"America/Chicago","license_code":"cc-by-nc","observed_time_zone":"America/Chicago","quality_metrics":[],"public_positional_accuracy":1018,"reviewed_by":[9677958],"oauth_application_id":2,"flags":[],"created_at":"2026-04-20T13:06:29-05:00","description":"Swamp","time_zone_offset":"-06:00","project_ids_with_curator_id":[],"observed_on":"2026-04-17","observed_on_string":"2026

import requests, zipfile, io
r = requests.get('http://www.inaturalist.org/places/inaturalist-places.csv.zip', stream=True)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall('/Users/isabellawoods/Documents/northwestern 25-26/CAPSTONE/DATA')

In [211]:
places = pd.read_csv('/Users/isabellawoods/Documents/northwestern 25-26/CAPSTONE/DATA/inaturalist-places.csv')

my_places = []
not_found = []

for park in parks:
    matches = places[places['display_name'].str.contains(park, case=False, na=False)]
    my_places.append(matches)
    if len(matches) == 0:
        not_found.append(park)

poi = pd.concat(my_places) #parks of interest
len(poi)

170

In [213]:
not_found

['Assateague Island National Seashore, US, MD',
 'Bandelier National Monument, US, NM',
 'Big South Fork National River and Recreation Area, US, TN',
 'Blue Ridge Parkway, US, VA',
 'Camp Nelson National Monument, US, KY',
 'Chesapeake and Ohio Canal National Historical Park, US, MD',
 'Craters of the Moon National Monument and Preserve, US, ID',
 'Delaware Water Gap National Recreation Area, US, PA',
 'Denali National Park and Preserve, US, AK',
 'Fort Donelson National Battlefield, US, TN',
 'Fort Larned National Historic Site, US, KS',
 'Fort Washington Park, US, MD',
 'Gateway National Recreation Area, US, NY',
 'George Washington Memorial Parkway, US, VA',
 'Glen Canyon National Recreation Area, US, UT',
 'Great Sand Dunes National Park and Preserve, US, CO',
 'Great Smoky Mountains National Park, US, TN',
 'Greenbelt Park, US, MD',
 'Harpers Ferry National Historical Park, US, WV',
 'Home of Franklin D. Roosevelt National Historic Site, US, NY',
 'Indiana Dunes National Park, US,

In [238]:
start_time = time.time()
base = 'https://api.inaturalist.org/v1/observations?place_id='

park_species = []

for i in range(len(poi)):
    place_id = poi.iloc[1]['id']
    place_name = poi.iloc[1]['display_name']
    url = f'{base}{place_id}&per_page=1'
        
    response = requests.get(url)
    
    # If you ran the API call and have the response:
    data = response.json()  # or json.loads(your_string)
    total_pages = math.ceil(data['total_results'] / 200)
    
    all_pages = []
    for page in range(1, total_pages + 1):
        url = f'{base}{place_id}&page={page}&per_page=200'
        response = requests.get(url)
        data = response.json()
        results = data['results']
        
        df = pd.json_normalize(results)
        cols = {
            'id':                          'obs_id',
            'uuid':                        'uuid',
            'observed_on':                 'observed_on',
            'quality_grade':               'quality_grade',
            'place_guess':                 'place_guess',
            'location':                    'location',
            'captive':                     'captive',
            'taxon.name':                  'ScientificName',
            'taxon.preferred_common_name': 'CommonName',
            'taxon.rank':                  'TaxonRank',
            'taxon.iconic_taxon_name':     'TaxonGroup',
            'taxon.native':                'native',
            'taxon.introduced':            'introduced',
            'user.login':                  'user',
        }
        
        df = df[cols.keys()].rename(columns=cols)
        
        # Split location into lat/lon vectorized (no loop)
        df[['latitude', 'longitude']] = df['location'].str.split(',', expand=True).astype(float)
        df = df.drop(columns='location')
    
        df['park_name'] = place_name
        time.sleep(1)
        all_pages.append(df)

    park_species.append(pd.concat(all_pages, ignore_index=True))
    
    end_time = time.time()
    print('Elapsed Time: ', (end_time - start_time)/60, 'minutes.')
iNaturalist = pd.concat(park_species, ignore_index = True)

KeyError: 'results'

In [240]:
print(data)

{'error': 'Result window is too large, page x size must be less than or equal to [10000]. Please narrow your search, or use a sliding window approach with id_above or id_below params.', 'status': 403}


In [242]:
start_time = time.time()
base = 'https://api.inaturalist.org/v1/observations?place_id='
park_species = []

cols = {
    'id':                          'obs_id',
    'uuid':                        'uuid',
    'observed_on':                 'observed_on',
    'quality_grade':               'quality_grade',
    'place_guess':                 'place_guess',
    'location':                    'location',
    'captive':                     'captive',
    'taxon.name':                  'ScientificName',
    'taxon.preferred_common_name': 'CommonName',
    'taxon.rank':                  'TaxonRank',
    'taxon.iconic_taxon_name':     'TaxonGroup',
    'taxon.native':                'native',
    'taxon.introduced':            'introduced',
    'user.login':                  'user',
}

for i in range(len(poi)):
    place_id = poi.iloc[i]['id']       # fixed: was hardcoded to iloc[1]
    place_name = poi.iloc[i]['display_name']  # fixed: was hardcoded to iloc[1]
    
    all_pages = []
    last_id = 0

    while True:
        url = f'{base}{place_id}&per_page=200&order=asc&order_by=id&id_above={last_id}'
        response = requests.get(url)

        if response.status_code != 200:
            print(f'[{place_name}] Error status {response.status_code}, retrying...')
            time.sleep(5)
            continue

        data = response.json()

        if 'results' not in data or len(data['results']) == 0:
            break

        results = data['results']

        df = pd.json_normalize(results)
        existing_cols = {k: v for k, v in cols.items() if k in df.columns}
        df = df[existing_cols.keys()].rename(columns=existing_cols)

        if 'location' in df.columns:
            df[['latitude', 'longitude']] = df['location'].str.split(',', expand=True).astype(float)
            df = df.drop(columns='location')

        df['park_name'] = place_name
        all_pages.append(df)

        last_id = results[-1]['id']
        time.sleep(1)

    park_df = pd.concat(all_pages, ignore_index=True) if all_pages else pd.DataFrame()
    park_species.append(park_df)

    end_time = time.time()
    print(f'[{i+1}/{len(poi)}] {place_name} — {len(park_df)} records | Elapsed: {(end_time - start_time)/60:.1f} min')

iNaturalist = pd.concat(park_species, ignore_index=True)
print(f'Total records: {len(iNaturalist)}')

[1/170] Abraham Lincoln Birthplace National Historical Park, US, KY — 816 records | Elapsed: 0.3 min


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
import math
import time
import requests
import pandas as pd

def fetch_with_retry(url, max_retries=5):
    """Fetch a URL with exponential backoff on failure."""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            if response.status_code == 200:
                return response.json()
            else:
                print(f'  Status {response.status_code}, retrying...')
        except (requests.exceptions.ConnectionError, 
                requests.exceptions.Timeout) as e:
            wait = 2 ** attempt  # 1, 2, 4, 8, 16 seconds
            print(f'  Connection error: {e}. Waiting {wait}s...')
            time.sleep(wait)
    print(f'  Failed after {max_retries} attempts: {url}')
    return None

start_time = time.time()
base = 'https://api.inaturalist.org/v1/observations?place_id='
park_species = []

cols = {
    'id':                          'obs_id',
    'uuid':                        'uuid',
    'observed_on':                 'observed_on',
    'quality_grade':               'quality_grade',
    'place_guess':                 'place_guess',
    'location':                    'location',
    'captive':                     'captive',
    'taxon.name':                  'ScientificName',
    'taxon.preferred_common_name': 'CommonName',
    'taxon.rank':                  'TaxonRank',
    'taxon.iconic_taxon_name':     'TaxonGroup',
    'taxon.native':                'native',
    'taxon.introduced':            'introduced',
    'user.login':                  'user',
}

for i in range(len(poi)):
    place_id = poi.iloc[i]['id']
    place_name = poi.iloc[i]['display_name']

    all_pages = []
    last_id = 0

    while True:
        url = f'{base}{place_id}&per_page=200&order=asc&order_by=id&id_above={last_id}'
        data = fetch_with_retry(url)

        if data is None or 'results' not in data or len(data['results']) == 0:
            break

        results = data['results']

        df = pd.json_normalize(results)
        existing_cols = {k: v for k, v in cols.items() if k in df.columns}
        df = df[existing_cols.keys()].rename(columns=existing_cols)

        if 'location' in df.columns:
            df[['latitude', 'longitude']] = df['location'].str.split(',', expand=True).astype(float)
            df = df.drop(columns='location')

        df['park_name'] = place_name
        all_pages.append(df)

        last_id = results[-1]['id']
        time.sleep(1.5)  # slightly longer sleep to be safer

    park_df = pd.concat(all_pages, ignore_index=True) if all_pages else pd.DataFrame()
    park_species.append(park_df)

    elapsed = (time.time() - start_time) / 60
    print(f'[{i+1}/{len(poi)}] {place_name} — {len(park_df)} records | Elapsed: {elapsed:.1f} min')

iNaturalist = pd.concat(park_species, ignore_index=True)
print(f'Total records: {len(iNaturalist)}')

[1/170] Abraham Lincoln Birthplace National Historical Park, US, KY — 816 records | Elapsed: 0.2 min
